In [ ]:
# es/python-101/hard/04-word-frequency
# Generated companion notebook for the PyDA course.
# Run cells top-to-bottom (or in any order) to follow the lesson.

print("PyDA — ready 🚀")


In [ ]:
# 💾 Load the course datasets into this environment
# The course data files live in the PyDA repo; pull them so
# `open("…")` / `pd.read_csv("…")` work exactly like on disk.
import os
def _fetch(name, aliases=()):
    if os.path.exists(name):
        return
    url = f"https://raw.githubusercontent.com/abderrahim-lectures/python-data-analysis-course/main/public/datasets/{name}"
    os.system(f"curl -sL -o {name} {url}")
    for alias in aliases:
        if not os.path.exists(alias):
            os.system(f"cp {name} {alias}")

_fetch("slm-corpus.csv", ())


Contar palabras

Una vez que tienes tokens, el siguiente paso es contar con qué frecuencia aparece cada palabra. Estos recuentos de frecuencia le dicen al modelo de lenguaje qué palabras son comunes (probables de aparecer en cualquier parte) y cuáles son raras (predictivas cuando aparecen).

## Conceptos clave

### Construir un dict de frecuencia

El patrón de conteo usa un dict donde cada clave es una palabra y el valor es su recuento. El método `get()` maneja el caso "es la primera vez que vemos esta palabra":


In [ ]:
def word_frequency(tokens):
    freq = {}
    for token in tokens:
        freq[token] = freq.get(token, 0) + 1
    return freq

tokens = ["the", "cat", "sat", "the", "dog", "sat", "the"]
freq = word_frequency(tokens)
print(freq)
# {'the': 3, 'cat': 1, 'sat': 2, 'dog': 1}


`freq.get(token, 0)` devuelve el recuento actual si la palabra existe, o `0` si es la primera vez que la vemos. Sumar 1 incrementa el recuento.

### El enfoque con defaultdict

Una alternativa usa `collections.defaultdict`, que crea automáticamente las claves faltantes:


In [ ]:
from collections import defaultdict

def word_frequency(tokens):
    freq = defaultdict(int)
    for token in tokens:
        freq[token] += 1
    return dict(freq)


Ambos enfoques producen el mismo resultado. La versión con `defaultdict` es un poco más limpia pero requiere un import.

### Estadísticas de vocabulario

Con un dict de frecuencia puedes calcular estadísticas útiles:


In [ ]:
freq = word_frequency(tokenize(full_text))

total_tokens = sum(freq.values())
unique_words = len(freq)

print(f"Total tokens: {total_tokens:,}")
print(f"Unique words: {unique_words:,}")
print(f"Vocabulary richness: {unique_words / total_tokens:.4f}")


**La riqueza de vocabulario** (únicas / totales) mide cuán diverso es el texto. Un valor cercano a 1.0 significa que casi todas las palabras son únicas; un valor cercano a 0.0 significa una repetición intensa.

### Las palabras más y menos frecuentes

Ordena el dict de frecuencia para encontrar los extremos:


In [ ]:
sorted_words = sorted(freq.items(), key=lambda item: item[1], reverse=True)

print("Top 10 words:")
for word, count in sorted_words[:10]:
    print(f"  {word}: {count}")

print("\nBottom 10 words:")
for word, count in sorted_words[-10:]:
    print(f"  {word}: {count}")


En la mayoría de los textos en inglés, "the", "of", "and", "to" y "a" dominan la cima de la lista. Esto sigue la **ley de Zipf** — la palabra más frecuente aparece aproximadamente el doble de veces que la segunda, el triple que la tercera, y así sucesivamente.

### Por qué importa la frecuencia para la generación

Un modelo de lenguaje usa la frecuencia para ponderar las predicciones. Si "the" aparece 500 veces y "platypus" aparece 2 veces, "the" debería elegirse con más frecuencia — pero no siempre. El modelo de bigramas refina esto condicionando sobre la palabra anterior, que es lo que hace que el texto generado sea legible en lugar de solo un flujo de "the the the".

## Inténtalo

Carga el corpus, tokenízalo y construye un dict de frecuencia. Luego responde:
1. ¿Cuántos tokens totales hay?
2. ¿Cuáles son las 5 palabras más frecuentes?
3. ¿Qué porcentaje del vocabulario consiste en palabras que aparecen solo una vez?


In [ ]:
texts = load_corpus("slm-corpus.csv")
full_text = " ".join(texts)
tokens = tokenize(full_text)
freq = word_frequency(tokens)

total = sum(freq.values())
hapax = sum(1 for w, c in freq.items() if c == 1)
print(f"Total tokens: {total}")
print(f"Words appearing once: {hapax} ({hapax/len(freq)*100:.1f}%)")


## Conclusiones clave

- `dict.get(clave, predeterminado)` es la base del conteo de frecuencias
- La riqueza de vocabulario (únicas / totales) mide la diversidad del texto
- Ley de Zipf: un pequeño número de palabras domina la distribución de frecuencias
- Los recuentos de frecuencia son la materia prima para las tablas de probabilidad de bigramas

## Reto de práctica

Escribe una función `top_n(freq, n)` que devuelva las N palabras más frecuentes como una lista de tuplas `(word, count)`. Luego úsala para encontrar las 20 palabras principales del corpus.


In [ ]:
def top_n(freq, n):
    return sorted(freq.items(), key=lambda item: item[1], reverse=True)[:n]


In [ ]:
# The end. Practice on your own — each cell is a minimal, runnable chunk.
